### Silver – menu_items

#### Purpose
Transform the Bronze `menu_items` table into a clean and analytics-ready
Silver table by:
- Standardizing column names using a reusable UDF
- Enforcing correct data types
- Handling nulls based on business rules
- Deduplicating records
- Isolating malformed records into a quarantine table

#### Source
- coffee.bronze.menu_items

#### Targets
- coffee.silver.menu_items
- coffee.silver.quarantine_menu_items


In [0]:
%python
# Widgets allow the same notebook to be executed across environments (DEV/PROD)
# and reused across multiple tables by changing only job parameters.
#
# default_watermark is used when the Silver table is empty (first run),
# enabling incremental ingestion logic without special casing.


dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("source_table", "menu_items")
dbutils.widgets.text("default_watermark", "1900-01-01")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
source_table = dbutils.widgets.get("source_table")
default_watermark = dbutils.widgets.get("default_watermark")

In [0]:
%run ./Silver_utils/silver_transform_utils


In [0]:
%python
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{source_table}")


In [0]:
%python
# Standardize all incoming column names using the central UDF.
# This ensures consistent snake_case naming across all Silver tables,
# regardless of how the raw files were named in Bronze.

df_std = standardize_columns(df_bronze)


In [0]:
%python
df_std.createOrReplaceTempView(f"bronze_{source_table}_std")


In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table} (
  item_id INT,
  item_name STRING,
  category STRING,
  price DOUBLE,
  is_seasonal BOOLEAN,
  available_from DATE,
  available_to DATE,

  -- Bronze metadata
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP,
  load_dt DATE,
  source_file STRING,
  source_table STRING,

  -- Silver audit
  silver_loaded_at TIMESTAMP,
  silver_updated_at TIMESTAMP
)
USING DELTA
""")


In [0]:
-- Incremental extraction:
-- Only process Bronze rows that arrived after the latest loaded_at timestamp
-- already present in the Silver target table.
--
-- This prevents reprocessing old Bronze records and keeps Silver rerun-safe.

CREATE OR REPLACE TEMP VIEW bronze_menu_items_incremental AS
SELECT *
FROM bronze_menu_items_std
WHERE loaded_at >
(
  SELECT COALESCE(MAX(loaded_at), '1900-01-01')
  FROM coffee.silver.menu_items
);


In [0]:
%python

#  Count invalid records for menu_items

# Required columns:
# item_id, item_name, category, price

invalid_count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM bronze_menu_items_incremental
WHERE
  item_id IS NULL
  OR item_name IS NULL
  OR category IS NULL
  OR price IS NULL
""").collect()[0]["cnt"]

print("Invalid menu_item rows:", invalid_count)


In [0]:
%python

# Data Quality Handling (Quarantine):
# Rows failing mandatory field validation are written to a quarantine table
# along with a quarantine_reason and quarantined_at timestamp.
# This prevents bad data from polluting Silver while still preserving it
# for debugging and auditability.

#  Create and load menu_items quarantine table only if invalid rows exist


if invalid_count > 0:

    
    #  Create menu_items quarantine table
    
    spark.sql("""
    CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table}_quarantine (
      item_id STRING,
      item_name STRING,
      category STRING,
      price STRING,
      is_seasonal STRING,

      -- Optional seasonal availability columns 
      available_from STRING,
      available_to STRING,

      -- Bronze metadata
      loaded_at TIMESTAMP,
      updated_at TIMESTAMP,
      load_dt DATE,
      source STRING,
      source_file STRING,

      -- Quarantine metadata
      quarantine_reason STRING,
      quarantined_at TIMESTAMP
    )
    USING DELTA
    """)

    
    #  Merge invalid rows into quarantine (idempotent)
    
    spark.sql("""
    MERGE INTO {catalog}.{silver_schema}.{source_table}_quarantine" q
    USING (
      SELECT
        *,
        CASE
          WHEN item_id IS NULL THEN 'item_id is null'
          WHEN item_name IS NULL THEN 'item_name is null'
          WHEN category IS NULL THEN 'category is null'
          WHEN price IS NULL THEN 'price is null'
          ELSE 'unknown validation failure'
        END AS quarantine_reason,
        current_timestamp() AS quarantined_at
      FROM bronze_menu_items_incremental
      WHERE
        item_id IS NULL
        OR item_name IS NULL
        OR category IS NULL
        OR price IS NULL
    ) b
    ON q.item_id = b.item_id
    AND q.quarantine_reason = b.quarantine_reason
    WHEN NOT MATCHED THEN
    INSERT *;
    """)

else:
    print("No invalid menu_items rows found. Quarantine table not created.")


In [0]:
%python
# merging data to silver table
spark.sql(f""" 
MERGE INTO  {catalog}.{silver_schema}.{source_table} s
USING (

  SELECT
    TRY_CAST(item_id AS INT)        AS item_id,
    item_name,
    category,
    TRY_CAST(price AS DOUBLE)       AS price,
    TRY_CAST(is_seasonal AS BOOLEAN) AS is_seasonal,
    TRY_CAST(available_from AS DATE) AS available_from,
    TRY_CAST(available_to AS DATE)   AS available_to,

    loaded_at,
    updated_at,
    load_dt,
    source_file,
    'coffee.bronze.menu_items' as source_table,

    current_timestamp() AS silver_updated_at
-- Deduplication logic:
-- Bronze may contain duplicates for the same business key.
-- We keep only the latest version of each record using:
--   ROW_NUMBER() OVER (PARTITION BY <business_key> ORDER BY updated_at DESC)
--
-- This ensures Silver contains a single clean record per business key.

  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY item_id
             ORDER BY updated_at DESC
           ) AS rn
    FROM bronze_menu_items_incremental
    WHERE
      item_id IS NOT NULL
      AND item_name IS NOT NULL
      AND category IS NOT NULL
      AND price IS NOT NULL
  )
  WHERE rn = 1

) b

ON s.item_id = b.item_id


-- UPDATE existing menu items
-- Idempotent MERGE into Silver:
-- - WHEN MATCHED: update existing rows with the latest values
-- - WHEN NOT MATCHED: insert new rows
--
-- This ensures rerun safety and supports incremental upserts.

WHEN MATCHED THEN
  UPDATE SET
    s.item_name          = b.item_name,
    s.category           = b.category,
    s.price              = b.price,
    s.is_seasonal        = b.is_seasonal,
    s.available_from     = b.available_from,
    s.available_to       = b.available_to,
    s.updated_at         = b.updated_at,
    s.load_dt            = b.load_dt,
    s.source_file        = b.source_file,
    s.source_table        = b.source_table,
    s.silver_updated_at  = b.silver_updated_at


-- INSERT new menu items

WHEN NOT MATCHED THEN
  INSERT (
    item_id,
    item_name,
    category,
    price,
    is_seasonal,
    available_from,
    available_to,
    loaded_at,
    updated_at,
    load_dt,
    source_file,
    source_table,
    silver_loaded_at,
    silver_updated_at
  )
  VALUES (
    b.item_id,
    b.item_name,
    b.category,
    b.price,
    b.is_seasonal,
    b.available_from,
    b.available_to,
    b.loaded_at,
    b.updated_at,
    b.load_dt,
    b.source_file,
    b.source_table,
    current_timestamp(),
    current_timestamp()
  )
  """)
